# Mini Project 1: Text Analysis
**Casey Collins (ccolli78)** — CS 445, Fall 2026

## Goal
Use word frequencies as a fingerprint of an author's style. I want to answer three questions:

1. Do two works by the **same author** have similar word-frequency profiles?
2. Do works by **two different authors** differ more than two works by the same author?
3. Which specific words does one author **prefer** that the other uses much less?

Extra credit: does the count of a common word such as *would* follow a binomial distribution
across chunks of a book, as it should if each word were an independent trial?

## Data sources
Plain-text novels from [Project Gutenberg](https://www.gutenberg.org), stored in `data/`:

| Author | Work | Gutenberg ID |
|---|---|---|
| Jane Austen | Pride and Prejudice | 1342 |
| Jane Austen | Emma | 158 |
| Charles Dickens | A Tale of Two Cities | 98 |
| Charles Dickens | Great Expectations | 1400 |

I picked two authors from the same century and country so that differences reflect style rather than era or dialect.

## Approach
1. Strip Gutenberg headers and footers, lowercase, tokenize on non-letters, drop stop words.
2. Build a relative word-frequency table (counts per 10,000 words) for each work.
3. Compare works with (a) rank correlation of shared-word frequencies, (b) Jaccard overlap of top-N vocabularies, and (c) scatter plots of frequencies.
4. Find preferred words with a log-ratio of smoothed frequencies between authors, and check them with a chi-square test.
5. Extra credit: split a book into fixed-size chunks, count *would* in each, and compare to a binomial fit.


## 1. Load and clean the texts
Each Gutenberg file has a licence header and footer marked by `*** START ...` and `*** END ...` lines. I keep only the body, then tokenize.

In [1]:
import re, os
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

WORKS = {
    'Austen: Pride and Prejudice': ('Austen',  'data/pg1342.txt'),
    'Austen: Emma':                ('Austen',  'data/pg158.txt'),
    'Dickens: A Tale of Two Cities': ('Dickens', 'data/pg98.txt'),
    'Dickens: Great Expectations':   ('Dickens', 'data/pg1400.txt'),
}

def load_gutenberg(path):
    # errors='replace' avoids unicode crashes on odd bytes (as noted in the class notebook)
    with open(path, encoding='utf-8', errors='replace') as f:
        raw = f.read()
    start = re.search(r'\*\*\* ?START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*', raw)
    end   = re.search(r'\*\*\* ?END OF (THE|THIS) PROJECT GUTENBERG EBOOK', raw)
    body = raw[start.end():end.start()] if start and end else raw
    return body

texts = {name: load_gutenberg(p) for name, (_, p) in WORKS.items()}
for name, t in texts.items():
    print(f'{name:32s} {len(t):>10,} characters')

Austen: Pride and Prejudice         728,751 characters
Austen: Emma                        880,434 characters
Dickens: A Tale of Two Cities       757,611 characters
Dickens: Great Expectations         994,623 characters


### Tokenizing
I lowercase everything and split on anything that is not a letter or apostrophe, so `don't` stays one token but `well-known` becomes two. Stop words (the, of, and ...) are removed for the vocabulary comparisons because they dominate every English text and hide stylistic differences. I keep a separate *unfiltered* token list for the extra-credit question, where function words like *would* are exactly what we want.

In [2]:
STOP = set(stopwords.words('english'))

def tokenize(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())

tokens_all = {name: tokenize(t) for name, t in texts.items()}          # everything
tokens     = {name: [w for w in toks if w not in STOP and len(w) > 1]  # content words only
              for name, toks in tokens_all.items()}

summary = pd.DataFrame({
    'author':        [WORKS[n][0] for n in WORKS],
    'total_tokens':  [len(tokens_all[n]) for n in WORKS],
    'content_tokens':[len(tokens[n]) for n in WORKS],
    'vocab_size':    [len(set(tokens[n])) for n in WORKS],
}, index=list(WORKS))
summary['type_token_ratio'] = (summary.vocab_size / summary.content_tokens).round(3)
summary

,author,total_tokens,content_tokens,vocab_size,type_token_ratio
Austen: Pride and Prejudice,Austen,128565,58063,6584,0.113
Austen: Emma,Austen,162099,73368,6954,0.095
Dickens: A Tale of Two Cities,Dickens,138490,64366,9544,0.148
Dickens: Great Expectations,Dickens,189051,83094,10594,0.127


### Relative frequencies
The books have different lengths, so raw counts are not comparable. I convert every count to **occurrences per 10,000 content words**.

In [3]:
counts = {name: Counter(toks) for name, toks in tokens.items()}
freq = pd.DataFrame(counts).fillna(0)
per10k = freq / freq.sum() * 10_000
per10k.round(2).sort_values(list(WORKS)[0], ascending=False).head(15)

,Austen: Pride and Prejudice,Austen: Emma,Dickens: A Tale of Two Cities,Dickens: Great Expectations
mr,139.16,157.29,96.63,85.57
elizabeth,111.09,1.09,0.00,0.00
could,91.45,114.08,43.81,58.13
would,83.53,111.90,53.29,72.09
darcy,74.40,0.00,0.00,0.00
said,69.92,65.97,102.69,162.35
mrs,60.97,95.55,2.80,19.74
bennet,58.38,0.00,0.00,0.00
much,58.04,66.24,28.74,37.43
miss,54.25,82.05,36.20,46.09
